## studentInfo(학생 정보) 데이터 체크(1차 전처리)


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

STUDENT_INFO_PATH = PROJECT_ROOT / "data" / "raw" / "studentInfo.csv"

print("프로젝트 위치:", PROJECT_ROOT)
print("studentInfo 존재:", STUDENT_INFO_PATH.exists())

프로젝트 위치: C:\SKN_AI\oulad-churn-prediction
studentInfo 존재: True


In [2]:
student_info = pd.read_csv(STUDENT_INFO_PATH)

print("student_info 크기:", student_info.shape)
display(student_info.head())

student_info 크기: (32593, 12)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [3]:
print("===== student_info 정보 =====")
student_info.info()

print("\nstudent_info 컬럼별 결측/공백/'?' 개수:")
for col in student_info.columns:
    none_count = (
        student_info[col].isnull()
        | (student_info[col].astype(str).str.strip() == "")
        | (student_info[col].astype(str).values == "?")
    ).sum()
    if none_count > 0:
        print(f"{col} : {none_count}")

===== student_info 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   code_module           32593 non-null  str  
 1   code_presentation     32593 non-null  str  
 2   id_student            32593 non-null  int64
 3   gender                32593 non-null  str  
 4   region                32593 non-null  str  
 5   highest_education     32593 non-null  str  
 6   imd_band              32593 non-null  str  
 7   age_band              32593 non-null  str  
 8   num_of_prev_attempts  32593 non-null  int64
 9   studied_credits       32593 non-null  int64
 10  disability            32593 non-null  str  
 11  final_result          32593 non-null  str  
dtypes: int64(3), str(9)
memory usage: 4.8 MB

student_info 컬럼별 결측/공백/'?' 개수:
imd_band : 1111


imd_band(지역 빈곤 지수 구간)에서만 1,111건의 결측치('?')가 확인된다.
대체할 뚜렷한 기준이 없는 범주형 값이므로, 해당 행은 제거하고 진행한다.


In [4]:
# imd_band 결측치('?') 행 제거
student_info_processed = student_info.drop(
    student_info[student_info["imd_band"] == "?"].index
).copy()

print("제거 전 행 수:", len(student_info))
print("제거 후 행 수:", len(student_info_processed))
print(
    "imd_band '?' 잔여 개수:",
    (student_info_processed["imd_band"] == "?").sum()
)

제거 전 행 수: 32593
제거 후 행 수: 31482
imd_band '?' 잔여 개수: 0


In [5]:
# imd_band 구간별 코드화 (예: '0-10%' -> '0')
def cat_imd(value):
    return value[0]

student_info_processed["imd_band_cd"] = (
    student_info_processed["imd_band"].apply(cat_imd)
)

student_info_processed = student_info_processed.drop(columns=["imd_band"])

print("imd_band_cd 종류:", sorted(student_info_processed["imd_band_cd"].unique()))

imd_band_cd 종류: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [6]:
# age_band 구간별 코드화
# 0-35 -> 'Y'(young), 35-55 -> 'M'(middle), 55<= -> 'S'(senior)
def cat_age(value):
    if value == "0-35":
        return "Y"
    elif value == "35-55":
        return "M"
    elif value == "55<=":
        return "S"
    else:
        return "ERR"

student_info_processed["age_band_cd"] = (
    student_info_processed["age_band"].apply(cat_age)
)

student_info_processed = student_info_processed.drop(columns=["age_band"])

print("age_band_cd 종류:", student_info_processed["age_band_cd"].unique())
print(
    "ERR로 분류된 행 수:",
    (student_info_processed["age_band_cd"] == "ERR").sum()
)

age_band_cd 종류: <ArrowStringArray>
['S', 'M', 'Y']
Length: 3, dtype: str
ERR로 분류된 행 수: 0


In [7]:
# highest_education 코드화
# 0: No Formal Quals, 1: Lower Than A Level, 2: A Level or Equivalent
# 3: HE Qualification, 4: Post Graduate Qualification
def cat_edu(value):
    match value:
        case "No Formal Quals":
            return "0"
        case "No Formal quals":
            return "0"
        case "Lower Than A Level":
            return "1"
        case "A Level or Equivalent":
            return "2"
        case "HE Qualification":
            return "3"
        case "Post Graduate Qualification":
            return "4"
        case _:
            return "ERR"

student_info_processed["highest_education_cd"] = (
    student_info_processed["highest_education"].apply(cat_edu)
)

student_info_processed = student_info_processed.drop(columns=["highest_education"])

print("highest_education_cd 종류:", student_info_processed["highest_education_cd"].unique())
print(
    "ERR로 분류된 행 수:",
    (student_info_processed["highest_education_cd"] == "ERR").sum()
)

highest_education_cd 종류: <ArrowStringArray>
['3', '2', '1', '4', '0']
Length: 5, dtype: str
ERR로 분류된 행 수: 0


In [8]:
STUDENT_INFO_KEYS = ["code_module", "code_presentation", "id_student"]

print("정제본 크기:", student_info_processed.shape)
print("\n정제본 결측치:")
print(student_info_processed.isna().sum())

print(
    "\n키 중복 행 수:",
    student_info_processed.duplicated(subset=STUDENT_INFO_KEYS).sum()
)

display(student_info_processed.head())

정제본 크기: (31482, 12)

정제본 결측치:
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
imd_band_cd             0
age_band_cd             0
highest_education_cd    0
dtype: int64

키 중복 행 수: 0


,code_module,code_presentation,id_student,gender,region,num_of_prev_attempts,studied_credits,disability,final_result,imd_band_cd,age_band_cd,highest_education_cd
0,AAA,2013J,11391,M,East Anglian Region,0,240,N,Pass,9,S,3
1,AAA,2013J,28400,F,Scotland,0,60,N,Pass,2,M,3
2,AAA,2013J,30268,F,North Western Region,0,60,Y,Withdrawn,3,M,2
3,AAA,2013J,31604,F,South East Region,0,60,N,Pass,5,M,2
4,AAA,2013J,32885,F,West Midlands Region,0,60,N,Pass,5,Y,1


In [9]:
# 저장 경로
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_INFO_OUTPUT_PATH = INTERIM_DIR / "student_info_processed.csv"

student_info_processed.to_csv(STUDENT_INFO_OUTPUT_PATH, index=False)

print("저장 완료")
print(
    STUDENT_INFO_OUTPUT_PATH.name,
    f"{STUDENT_INFO_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB"
)

저장 완료
student_info_processed.csv 1.69 MB
